In [ ]:
#!pip install ipywidgets datasets cryptography torch transformers sentencepiece matplotlib wordcloud


In [ ]:
##Namen des Chats bekommen##

import re

def getNames(f):
    
    
    pattern = r"\d{2}\.\d{2}\.\d{2}, \d{2}:\d{2} - (.*?):"

    names = set()

   
    for line in f:
        match = re.search(pattern, line)
        if match:
            names.add(match.group(1))
    return names



In [ ]:
#Fehlerfilter1 - Nachrichten und Anrufe sind Ende-zu-Ende-verschlüsselt. Nur Personen in diesem Chat können sie lesen, anhören oder teilen. Mehr erfahren
import re
from pathlib import Path

INPUT_DIR = Path("./data/Chats")  ###Ordner der aus Whatsapp exportierten Chats
OUTPUT_DIR = Path("./data/Chats2") ###Ausgabeordner für gefilterte Chas
REMOVED_FILE = Path("./data/removed_Chat2.txt") ###Kontrollordner für Zwischenspeicherung aller Änderungen

OUTPUT_DIR.mkdir(exist_ok=True)

# Header-Pattern für leere Nachrichten
HEADER_PATTERN = re.compile(r"^\d{2}\.\d{2}\.\d{2}, \d{2}:\d{2} - [^:]+:")

#Ganze Zeile löschen
def is_ignorable_system_message(line: str) -> bool:
    system_phrases = [
        "ist ein Kontakt.",
        "Nachrichten und Anrufe sind Ende-zu-Ende-verschlüsselt",
        "Warte auf diese Nachricht",
        "Nachricht wurde gelöscht",
        "Du hast diese Nachricht gelöscht",
    ]
    return any(phrase in line for phrase in system_phrases)

# Ausschneiden "<Diese Nachricht wurde bearbeitet.>"
def remove_edit_marker(line: str) -> str:
    return re.sub(r"\s*<[^>]*bearbeitet[^>]*>", "", line)

#anonymisieren Nummern, Telefonnummern, IBANs 
def mask_sensitive_numbers(line: str) -> str:
    #IBAN
    line = re.sub(r"\b[A-Z]{2}\d{2}[A-Z0-9]{10,32}\b", "[IBAN]", line)

    #Telefonnummern / lange Zahlen
    line = re.sub(r"\b\d{6,}\b", "[NUMBER]", line)

    #Zahlen mit Leerzeichen oderBindestrichen
    line = re.sub(r"\b(?:\+?\d[\d \-]{7,}\d)\b", "[PHONE]", line)

    return line

#Leere Nachrichten rausfiltern 
def is_empty_message_line(line: str) -> bool:
    if not HEADER_PATTERN.match(line):
        return False  # keine WhatsApp Header
    # Text hinter "Name:" 
    content = line.split(":", 1)[1].strip()
    return content == "" or content in {"***", "<Medien ausgeschlossen>"}  

with REMOVED_FILE.open("w", encoding="utf-8") as removed_out:
    for in_path in INPUT_DIR.glob("*.txt"):
        out_path = OUTPUT_DIR / in_path.name

        removed_count = 0
        total_count = 0

        with in_path.open("r", encoding="utf-8") as fin, out_path.open("w", encoding="utf-8") as fout:
            for line in fin:
                total_count += 1

                #Systemnachricht löschen
                if is_ignorable_system_message(line):
                    removed_count += 1
                    removed_out.write(f"[{in_path.name}] [SYSTEM] {line}")
                    continue

                #Bearbeiten-Nachricht entfernen
                cleaned = remove_edit_marker(line)

                #Zahlen zensieren
                cleaned = mask_sensitive_numbers(cleaned)

                #Leere Nachrichten entfernen
                if is_empty_message_line(cleaned):
                    removed_count += 1
                    removed_out.write(f"[{in_path.name}] [EMPTY] {cleaned}")
                    continue

                fout.write(cleaned)

        print(f"{in_path.name}: {removed_count}/{total_count} Zeilen entfernt/maskiert -> {out_path.name}")


In [ ]:
#Klasse für alle Nachrichten 
import datetime

class Message:
    
    def __init__(self, chatName, sender, text, when, trustlevel):
        self.name = chatName
        self.sender = sender 
        self.receiver = "Mau" if sender == chatName else chatName
        self.text = text
        self.when = datetime.datetime.strptime(when, "%d.%m.%y, %H:%M")
        self.trustlevel = trustlevel
        #self.message_text()
        
    #def message_text(self):
        #Eigene Veränderungen finden
        #self.text = self.text.replace('\n','  ').replace(':',';').replace('#',';')
        #if self.text == '': self.text = '***'
    
    #def should_exclude(self):
        ###Bedingung für Ausschluss der Nacchricht

In [ ]:
###Datenerhebung###
#Namen austauschen, Beziehungsstatus#

#### Zensiert ####

#401 = Yoda = Volles Vertrauen
#401 = Ashoka = Volles Vertrauen
#401 = Plokoon = Sehr vertraut
#401 = Anakin = Volles Vertrauen
#401= Luminara = Volles Vertrauen
#401 = Kitfisto = Sehr vertraut
#401 = Sifodyas = Sehr vertraut
#401 = Lando = Sehr vertraut
#401 = Luke = Sehr vertraut
#401 = Han = Volles Vertrauen
#401 = Padme = Sehr vertraut
#401 = Leia = Sehr vertraut
#401 = Yaddle = Sehr vertraut
#401 = Jango = Sehr vertraut
#401 = Bail = durchschnittliches Vertrauen
#401 = Chewie = Sehr vertraut
#401 = Mas = Sehr vertraut
#401 = Luthen = Sehr vertraut
#401 = Aayla = Sehr vertraut
#401 = Shaakti = Durchschnittliches Vertrauen
#401 = Dexter = Durchschnittliches Vertrauen
#401 = Niennumb = Sehr vertraut
#401 = Babufrik = Sehr vertraut
#401 = Silas = Durchschnittliches Vertrauen
#401 = Fox = Durchschnittliches Vertrauen
#401 = Hego = Formell/ etwas vertrauen
#401 = Rex = Sehr vertraut
#401 = Wolff = Durchschnittliches Vertrauen
#401 = Aura = Formell/ etwas vertrauen
#401 = Cody = Formell/ etwas vertrauen
#401 = Wedge = Formell
#401 = Rey = Formell

#Vertrauenlevel 
#Volles Vertrauen = 5
#Sehr vertraut = 4
#durchschnittliches Vertrauen = 3
#Formell/ etwas vertrauen = 2
#Formell = 1

def hideName(realName):
    match realName:
        case "401":
            return "401"
        case "401":
            return "Yoda"
        case "401":
            return "Ahsoka"
        case "401":
            return "Plokoon"
        case "401":
            return "Anakin"
        case "401":
            return "Luminara"
        case "401":
            return "Kitfisto"
        case "401":
            return "Sifodyas"
        case "401":
            return "Lando"
        case "401":
            return "Luke"
        case "401":
            return "Han"
        case "401":
            return "Padme"
        case "401":
            return "Leia"
        case "401":
            return "Yaddle"
        case "401":
            return "Jango"
        case "401":
            return "Bail"
        case "401":
            return "Chewie"
        case "401":
            return "Mas"
        case "401":
            return "Luthen"
        case "401":
            return "Aayla"
        case "401":
            return "Shaakti"
        case "401":
            return "Dexter"
        case "401":
            return "Niennumb"
        case "401":
            return "Babufrik"
        case "401":
            return "Silas"
        case "401":
            return "Fox"
        case "401":
            return "Hego"
        case "401":
            return "Rex"
        case "401":
            return "Wolff"
        case "401":
            return "Aura"
        case "401":
            return "Cody"
        case "401":
            return "Wedge"
        case "401":
            return "Rey"
        
def getTrustlevel(realName):
    match realName:
        case "401":
            return 5
        case "401":
            return 5
        case "401":
            return 4
        case "401":
            return 5
        case "401":
            return 5
        case "401":
            return 4
        case "401":
            return 4
        case "401":
            return 4
        case "401":
            return 4
        case "401":
            return 5
        case "401":
            return 4
        case "401":
            return 4
        case "401":
            return 4
        case "401":
            return 4
        case "401":
            return 3
        case "401":
            return 4
        case "401":
            return 4
        case "401":
            return 4
        case "401":
            return 4
        case "401":
            return 3
        case "401":
            return 3
        case "401":
            return 4
        case "401":
            return 4
        case "401":
            return 3
        case "401":
            return 3
        case "401":
            return 2
        case "401":
            return 4
        case "401":
            return 3
        case "401":
            return 2
        case "401":
            return 2
        case "401":
            return 1
        case "401":
            return 1

In [ ]:
#Nachrichten zählen#

import re, os

chatDirectory = ".data/chats2"
chatPaths = os.listdir(chatDirectory)

chatMetaData = []

headerPattern = re.compile(r"^(\d{2}\.\d{2}\.\d{2}), (\d{2}:\d{2}) - ([^:]+):")

allMessages = []
failMessages = []

for chatPath in chatPaths:
    
    msgCount = 0
    filteredMsgCount = 0
    medienCount = 0
    urlCount = 0
    filepath = os.path.join(chatDirectory, chatPath)
    
    trustlevel = 0
    
    # Namen holen 
    with open(filepath, "r", encoding="utf-8") as f:
        chatNames = getNames(f)
        
    with open(filepath, "r", encoding="utf-8") as f:
        chat = f.read()
    
    chatName = chatNames.pop()
    if "Mau" in chatName:
        print(chatName)
        chatName = chatNames.pop()
        print(chatName)
    trustlevel = getTrustlevel(chatName)
    chatName = hideName(chatName)
    print(chatName)
    
    # Split message
    chatLines = re.split(
        r"(^\d{2}\.\d{2}\.\d{2}, \d{2}:\d{2} - [^:]+:)",
        chat,
        flags=re.MULTILINE
    )
    
    
    

    #Chat iterieren
    for i in range(1, len(chatLines), 2):
        header = chatLines[i].strip()
        body = chatLines[i+1] if i + 1 < len(chatLines) else ""

        #FILTER
        #Medien zählen 
        if "<Medien ausgeschlossen>" in body:
            msgCount+=1
            medienCount += 1
            continue
            
        #URLs zählen 
        if "https://" in body:
            msgCount += 1
            urlCount += 1
            continue

        match = headerPattern.match(header)
        if not match:
            failMessages.append(header)
            continue

        date_str, time_str, sender = match.groups()
        text = body.strip()  #kompletter Text
        when = f"{date_str}, {time_str}"

        allMessages.append(Message(chatName, hideName(sender), text, when, trustlevel))
        msgCount += 1
        filteredMsgCount += 1
   
    #Metadaten pro Chat
    medienRel = medienCount / msgCount if msgCount > 0 else 0
    urlRel = urlCount / msgCount if msgCount > 0 else 0
    chatMetaData.append([chatName, msgCount, filteredMsgCount, medienCount, round(medienRel, 3), urlCount, round(urlRel, 3), trustlevel])

#Sortieren und output
print("Name, Nachrichten, Nachrichten (Filter), Medien, Medienverhältnis, URLs, URLverhältnis, Vertrauenslevel")

chatMetaData.sort(reverse=True, key=lambda x: x[1])
for chatSorted in chatMetaData:
    print(chatSorted)


In [ ]:
print(len(failMessages))
print(len(allMessages))

In [ ]:
#Analyse Nachrichten nach Vertrauenslevel Plot

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

from collections import Counter

trust_counts = Counter(msg.trustlevel for msg in allMessages)
print(trust_counts)

#X = Trust-Werte (1–5)
levels = sorted(trust_counts.keys())

#Y = Anzahl Nachrichten pro Level
counts = [trust_counts[level] for level in levels]

plt.figure(figsize=(8, 5))
plt.bar(levels, counts)

plt.xlabel("Trust-Level")
plt.ylabel("Anzahl Nachrichten")
plt.title("Nachrichten pro Trust-Level")

plt.xticks(levels)  

plt.show()

In [ ]:
#Plot - Nachrichten pro Chat

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

#Nachrichten insgesamt
messagesTotal = 0
medienTotal = 0
medienRelation = 0

namesList = []
msgCount = []

for chatData in chatMetaData:
    
    messagesTotal += chatData[1]
    medienTotal += chatData[3]
    
    namesList.append(chatData[0])
    msgCount.append(chatData[1])
    
print(messagesTotal)
print(medienTotal)
print(medienTotal/ messagesTotal)

print(f'A total of {len(allMessages):,} messages')



x = np.array(msgCount)
y = np.array(namesList)

plt.xlabel("Nachrichten Anzahl")
plt.ylabel("Sender")

plt.title("Anzahl aller Nachrichten")

plt.barh(y,x, height = 0.5)
plt.show()

In [ ]:
def group_consecutive_messages(messages):
    
    #Zusammenfassen von Nachrichten, wenn Sender und Empfänger zusammen gehören
    #Die Nachrichten sind erst ab hier nach den zugehörigen Chats sortiert
    grouped = []

    current_sender = None
    buffer = []
    current_trustlevel = None
    current_when = None

    for msg in messages:
        if msg.sender != current_sender:
            if buffer:
                grouped.append({
                    "sender": current_sender,
                    "text": "\n".join(buffer),
                    "trustlevel": current_trustlevel,
                    "when": current_when
                })

            current_sender = msg.sender
            buffer = [msg.text]
            current_trustlevel = msg.trustlevel
            current_when = msg.when
        else:
            buffer.append(msg.text)

    if buffer:
        grouped.append({
            "sender": current_sender,
            "text": "\n".join(buffer),
            "trustlevel": current_trustlevel,
            "when": current_when
        })

    return grouped

In [ ]:
allMessages.sort(key=lambda x: x.name)
%store allMessages
group_consecutive_messages(allMessages)

In [ ]:
#Kontextstärke der Nachrichten wird hier bestimmt
#Dialoglänge wird bestimmt
#Je mehr Kontext desto weniger einzelne Daten im Datenpool

def render_context(blocks, me="Mau"):
    
    #User - Assistant Format
    lines = []

    for block in blocks:
        text = str(block["text"]).strip()
        if not text:
            continue

        if block["sender"] == me:
            lines.append(f"Assistant: {text}")
        else:
            lines.append(f"User: {text}")

    return "\n".join(lines)


def make_context_training_pairs(
    grouped_messages,
    me="Mau",
    max_context_blocks=10,
    min_context_blocks=1,
    require_previous_user=True
):

    pairs = []

    for i, curr in enumerate(grouped_messages):
        #Nachricht muss von mir kommen
        if curr["sender"] != me:
            continue

        #Es muss Kontext davor geben
        if i == 0:
            continue

        #direkt vorher muss der andere geschrieben haben
        if require_previous_user and grouped_messages[i - 1]["sender"] == me:
            continue

        start = max(0, i - max_context_blocks)
        context_blocks = grouped_messages[start:i]

        if len(context_blocks) < min_context_blocks:
            continue

        context = render_context(context_blocks, me=me).strip()
        target = str(curr["text"]).strip()

        if not context or not target:
            continue

        pairs.append({
            "trust": curr["trustlevel"],
            "input": context + "\nAssistant:",
            "output": target,
            "text": context + "\nAssistant: " + target,
            "when": curr["when"],
            "context_blocks": len(context_blocks)
        })

    return pairs


grouped = group_consecutive_messages(allMessages)

groupedmessages = make_context_training_pairs(
    grouped,
    me="Mau",
    max_context_blocks=8,
    min_context_blocks=1,
    require_previous_user=True
)

print("Anzahl Trainingsbeispiele:", len(groupedmessages))
print(groupedmessages[1]["text"])

In [ ]:
#Plot Verteilung der Nachrichten nach Vertrauenslevel

import pandas as pd
import matplotlib.pyplot as plt


df = pd.DataFrame(groupedmessages)

#Anzahl je Trustlevel
counts = df["trust"].value_counts().sort_index()

plt.figure(figsize=(8,4))
plt.bar(counts.index.astype(str), counts.values)
plt.xlabel("Trustlevel")
plt.ylabel("Anzahl")
plt.title("Verteilung der Trustlevel (absolut)")
plt.tight_layout()
plt.show()


In [ ]:
#Plot Verteilung der Nachrichten prozentual nach Vertrauenslevel
percent = (df["trust"].value_counts(normalize=True).sort_index() * 100)

plt.figure(figsize=(8,4))
plt.bar(percent.index.astype(str), percent.values)
plt.xlabel("Trustlevel")
plt.ylabel("Anteil (%)")
plt.title("Verteilung der Trustlevel (in %)")
plt.tight_layout()
plt.show()


In [ ]:
#Datensatz bereinigen und in verschiedene Datensätze aufteilen

import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset


#1 Gesamten Datensatz laden

df = pd.DataFrame(groupedmessages).copy()


#2 Daten Clean machen

df["input"] = df["input"].astype(str)
df["output"] = df["output"].astype(str)
df["text"] = df["text"].astype(str)

df = df.dropna(subset=["trust", "input", "output", "text"])

df = df[
    (df["input"].str.strip().str.len() >= 1) &
    (df["output"].str.strip().str.len() >= 1) &
    (df["text"].str.strip().str.len() >= 1)
]

#Zu lange Beispiele entfernen
df = df[df["text"].str.len() <= 3000]

df = df.reset_index(drop=True)

print("Gesamter Datensatz:", len(df))
print("Trust-Verteilung:")
print(df["trust"].value_counts().sort_index())

if "context_blocks" in df.columns:
    print("Kontextblöcke:")
    print(df["context_blocks"].describe())


#3 Test/Evaluation Split 


eval_size = 100

train_val_df, test_df = train_test_split(
    df,
    test_size=eval_size,
    random_state=42,
    stratify=df["trust"] if "trust" in df.columns else None
)

train_val_df = train_val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train+Val:", len(train_val_df))
print("Test/Evaluation:", len(test_df))

print("\nTest Trust-Verteilung:")
print(test_df["trust"].value_counts().sort_index())


#4 Train/Validation Split 


train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.1,
    random_state=42,
    stratify=train_val_df["trust"] if "trust" in train_val_df.columns else None
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("\nTrain:", len(train_df))
print("Val:", len(val_df))

print("\nTrain Trust-Verteilung:")
print(train_df["trust"].value_counts().sort_index())

print("\nVal Trust-Verteilung:")
print(val_df["trust"].value_counts().sort_index())


#5 Hugging Face Datensatz aus DataFrame


def gen_records(df):
    for _, row in df.iterrows():
        yield {
            "text": str(row["text"]).strip(),
            "trust": int(row["trust"]),
            "context_blocks": int(row["context_blocks"]) if "context_blocks" in row and pd.notna(row["context_blocks"]) else None,
        }

train_ds = Dataset.from_generator(lambda: gen_records(train_df))
val_ds = Dataset.from_generator(lambda: gen_records(val_df))

print(train_ds[1]["text"])


#6 Testdatensatz für Evaluation exportieren


eval_df = test_df.copy()
eval_df = eval_df.rename(columns={"output": "reference"})

eval_df = eval_df.reset_index(drop=True)
eval_df["eval_id"] = range(len(eval_df))

eval_df.to_csv("eval_set_100.csv", index=False, encoding="utf-8-sig")

print(eval_df[["eval_id", "input", "reference", "trust"]].head())

In [ ]:
#del df, low, high, pilot_df
import gc
gc.collect()

In [ ]:
#del train_df, val_df
gc.collect()

In [ ]:
#Versionen checken
import sys, torch, transformers
print("python:", sys.executable)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)


In [ ]:
#Model vorbereiten und Tokenizer laden
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = r"c:/401\llama-3.2-3b-hf" #3B-Instruct #3b-hf

tok = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="cuda", #nochmal auf CUDA umstellen
    dtype=torch.float16,
    local_files_only=True
)



In [ ]:
#Model überprüfen
print(type(tok))
print(model.__class__.__name__)
print(next(model.parameters()).device)


In [ ]:
#Verwendete Klassen überprüfen
import trl, inspect
print("trl:", trl.__version__)
print("SFTTrainer args:", list(inspect.signature(__import__('trl').SFTTrainer.__init__).parameters.keys()))


In [ ]:
#Zur Reduzierung der Speicherauslastung werden lange Tokenketten gefiltert
#Falls Training mit 512 statt 1024

def too_long(example):
    return len(tok(example["text"], add_special_tokens=False)["input_ids"]) <= 500

train_ds = train_ds.filter(too_long)
val_ds   = val_ds.filter(too_long)

In [ ]:
#Testet wie viele bei 512 Daten abschneidet

def token_len(example):
    return len(tok(example["text"], add_special_tokens=False)["input_ids"])

lengths = [token_len(x) for x in groupedmessages]

print("Durchschnitt:", sum(lengths) / len(lengths))
print("Max:", max(lengths))
print("Über 512:", sum(l > 512 for l in lengths))

In [ ]:
#Grafikkarte überprüfen
import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("tok.model_max_length:", tok.model_max_length)


In [ ]:
#Speicherübersicht, um die Speicherauslastung zu kontrollieren
import torch
import gc

def show_mem(tag=""):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        print(f"\n--- {tag} ---")
        print(f"allocated:      {torch.cuda.memory_allocated()/1024**3:.2f} GB")
        print(f"reserved:       {torch.cuda.memory_reserved()/1024**3:.2f} GB")
        print(f"max allocated:  {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
        print(f"max reserved:   {torch.cuda.max_memory_reserved()/1024**3:.2f} GB")
        print(torch.cuda.memory_summary())
        
torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

torch.cuda.memory._record_memory_history()

show_mem("start")

In [ ]:
#löscht ungenutzte Speichernutzung und aktualisiert Speicherverbrauch

from transformers import TrainerCallback
import gc, torch, os, psutil

process = psutil.Process(os.getpid())

class CleanupCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 50 == 0 and state.global_step > 0:
            gc.collect()
            torch.cuda.empty_cache()
            rss = process.memory_info().rss / 1024**3
            print(f"[cleanup step {state.global_step}] RSS RAM: {rss:.2f} GB")

trainer.add_callback(CleanupCallback())

In [ ]:
#Hier passiert Feintuning Magie!

def formatting_func(example):
    return example["text"]

tok.model_max_length = 512

from peft import LoraConfig
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

#LoRA Konfiguration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj"]
)
#Trainingskonfigurationen
args = SFTConfig(
    output_dir="out_pilot_###3b", #3b-instruct #3b
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_steps=2000,          # Pilot= 200
    warmup_steps=20,
    logging_steps=10,        #10
    eval_strategy="steps",      #steps
    eval_steps=50,
    #save_strategy="no",
    save_steps=50,
    save_total_limit=1,
    fp16=True,
    report_to="none",
    #packing=False,
    max_length=512,
    
    #RAM Einstellungen, ggf Platz sparen
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    dataloader_persistent_workers=False,
    remove_unused_columns=True,
    
)

#Trainingskomponenten
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tok,          
    peft_config=lora_config,
    formatting_func=formatting_func,
    
)

trainer.add_callback(CleanupCallback())
trainer.train(resume_from_checkpoint="out_pilot_###3b/checkpoint-1850") ###Von Checkpoint fortsetzen
#trainer.train()                                                        ###Neustart des Trainings

In [ ]:
model.unload()

In [ ]:
show_mem("after Training")

In [ ]:
import sys, site
print("python:", sys.executable)
print("user site:", site.getusersitepackages())

In [ ]:
test_prompts = [
    "User: Ey sorry hab gestern voll vergessen zu antworten 😅\nAssistant:",
    "User: Alles gut bei dir heute?\nAssistant:",
    "User: Haha ja stimmt, war echt lustig gestern\nAssistant:",
    "User: Ich weiß noch nicht, ob ich Zeit hab 🤔\nAssistant:",
    "User: Danke dir fürs Bescheid sagen!\nAssistant:",
    "User: Bin grad mega gestresst wegen Uni 😩\nAssistant:",
    "User: Wollen wir das morgen klären?\nAssistant:",
    "User: Ich glaub das wird knapp heute\nAssistant:",
    "User: Ey das war echt nicht cool von dir…\nAssistant:",
    "User: Gute Nacht 😊\nAssistant:",
]


In [ ]:
#Neue Instanz des Modells laden

def generate_only_new(model, prompt, max_new_tokens=80, temperature=0.7):
    model.eval()
    inputs = tok(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            eos_token_id=tok.eos_token_id,
            pad_token_id=tok.eos_token_id,
        )

    cut = inputs["input_ids"].shape[-1]
    gen_ids = out[0][cut:]
    return tok.decode(gen_ids, skip_special_tokens=True).strip()


In [ ]:
#10 Inferenzen pro Modell generieren

results = []

#---------- BASE ----------
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="cuda",
    offload_folder="offload",
    dtype=torch.float16,
    local_files_only=True
)

for p in test_prompts:
    base_out = generate_only_new(base_model, p)
    results.append({
        "prompt": p,
        "base": base_out,
        "ft": None
    })

del base_model
gc.collect()
torch.cuda.empty_cache()

#---------- FINE-TUNED ----------
base_cpu = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map={"": "cpu"},
    dtype=torch.float16,
    local_files_only=True
)

ft = PeftModel.from_pretrained(base_cpu, adapter_path)
ft_merged = ft.merge_and_unload().eval().to("cuda")

for i, p in enumerate(test_prompts):
    ft_out = generate_only_new(ft_merged, p)
    results[i]["ft"] = ft_out

del base_cpu, ft, ft_merged
gc.collect()
torch.cuda.empty_cache()


In [ ]:
show_mem("after model load 3")

In [ ]:
#torch.cuda.memory._dump_snapshot("cuda_snapshot.pickle")

In [ ]:
torch.cuda.empty_cache()